# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding #2 — "The Content Performance Curve"** (paper page 7): health score rises through
content age, peaking at 61-90 days (health 33.1), then falls to a low of 14.1 at 271-365 days,
with a partial recovery at 365+ days (25.1) attributed to refreshed older pages.

**My methodology question:** this compares *different pages of different ages* at one point in
time (cross-sectional), not the *same pages tracked over time* (longitudinal). A true lifecycle
claim — "content declines as it ages" — implies a within-page trajectory. But an alternative,
equally consistent explanation is a cohort effect: pages published 9-12 months ago might simply
have been created under different editorial standards, targeting different (harder) keywords,
than pages published last month. The paper itself is careful to call the 365+ recovery
"actionable" rather than automatic, which is the right instinct — I'd ask whether the same
caution should extend to the whole curve, not just the recovery tail.

---

**Finding #4 — "The Freshness Multiplier"** (paper page 9): 365+ day content that was refreshed
within 30 days shows a 3.2x health boost (10.7 → 34.5) and 57x more impressions (71 → 4,039).

**My methodology question:** which pages get refreshed is a choice, not random — an editor or
a workflow rule likely selects pages worth the effort (already has some traffic, a fixable
issue, business priority), rather than refreshing a random sample of old pages. If so, part of
the 3.2x/57x gap may reflect *which pages get chosen* for refresh, not the refresh itself. I'd
ask: was refresh assignment closer to random, or was it based on a signal (like existing
impressions) that would predict recovery even without the update? This doesn't mean the finding
is wrong — refresh may well help — just that isolating "how much" would need either a randomized
test or a matched comparison (refreshed vs. similar-but-not-yet-refreshed pages).

In [1]:
# Numbers cited above, recorded here for the record (no client/private data — public paper stats)
paper_findings = {
    'finding_2_content_curve': {
        'health_61_90d': 33.1, 'health_271_365d': 14.1, 'health_365plus': 25.1,
        'source': 'FlyRank State of AI-Driven SEO, March 2026, p.7'
    },
    'finding_4_freshness_multiplier': {
        'health_boost_refreshed_365plus': '3.2x (10.7 -> 34.5)',
        'impression_boost_refreshed_365plus': '57x (71 -> 4039)',
        'source': 'FlyRank State of AI-Driven SEO, March 2026, p.9'
    }
}
for k, v in paper_findings.items():
    print(k, '->', v)


finding_2_content_curve -> {'health_61_90d': 33.1, 'health_271_365d': 14.1, 'health_365plus': 25.1, 'source': 'FlyRank State of AI-Driven SEO, March 2026, p.7'}
finding_4_freshness_multiplier -> {'health_boost_refreshed_365plus': '3.2x (10.7 -> 34.5)', 'impression_boost_refreshed_365plus': '57x (71 -> 4039)', 'source': 'FlyRank State of AI-Driven SEO, March 2026, p.9'}


## 2. My model under an honest split (before/after)

**Before/after: does a naive random split overstate model performance compared to the honest
GroupKFold-by-client split already used in ML-08?**

"Before" = 5-fold random KFold (ignores that content items from the same client might share
hidden site-level traits). "After" = the same GroupKFold-by-client split from ML-08, reused
here for direct comparison. Same model (Random Forest), same features, same metric.
**Results:**

| Split strategy | MAE | R² | Spearman |
|---|---|---|---|
| BEFORE: random KFold (naive, seed=42) | 0.008635 | 0.013 | -0.012 |
| BEFORE: random KFold (naive, seed=7, robustness check) | 0.008617 | 0.013 | -0.008 |
| AFTER: GroupKFold by client (honest) | 0.007734 | -0.028 | 0.058 |

**This is a genuinely mixed result, and I want to report it honestly rather than force it into
the "naive split looks artificially good" story I expected going in.**

Confirmed stable across two random seeds (not a fluke): the naive split has *worse* MAE than
the honest split (0.0086 vs 0.0077), and *worse* rank correlation, but *better* R² (0.013 vs
-0.028, i.e. positive vs negative).

**My interpretation:** R² being higher under the naive split is consistent with what leakage
usually looks like — the model gets a slight assist from seeing other pages of the same client
in training, inflating R² specifically. But MAE and Spearman moving in the *opposite* direction
is unexpected, and I don't have a fully confident explanation for it. My best guess: with this
particular dataset (46 clients, fairly even page counts per client), a random split still keeps
enough same-client pages out of each test fold by chance that the "memorize the client" effect
is weak — so the main thing distinguishing the two splits here may just be different specific
rows landing in each fold, not a strong leakage effect. I'm reporting this as an honest, partly
unresolved finding rather than overstating a clean "leakage confirmed" story the R² number alone
would suggest.

**Confirmed clean regardless:** `client_hash_id` was never included as a feature in either
version — the difference is purely about which rows go into train vs. test, not about the
model literally seeing the client identity.

In [2]:

%pip -q install duckdb huggingface_hub scikit-learn
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

data = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        SUM(f.gsc_clicks)::DOUBLE / NULLIF(SUM(f.gsc_impressions), 0) AS ctr_observed,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS impressions_total,
        ANY_VALUE(d.word_count) AS word_count,
        ANY_VALUE(d.content_type) AS content_type,
        ANY_VALUE(d.search_volume) AS search_volume,
        ANY_VALUE(d.competition) AS competition,
        DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS age_days
    FROM {FACT} f
    JOIN {DIM_CONTENT} d ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE AND d.is_deleted IS NOT TRUE
    GROUP BY f.content_hash_id, f.client_hash_id
""").df().dropna(subset=['ctr_observed', 'avg_position', 'word_count', 'age_days'])

data['search_volume'] = data['search_volume'].astype('float64').fillna(0)
data['competition'] = data['competition'].astype('float64').fillna(0)

from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr

X_cols_numeric = ['avg_position', 'word_count', 'search_volume', 'competition', 'age_days']
X_cols_categorical = ['content_type']
X = data[X_cols_numeric + X_cols_categorical]
y = data['ctr_observed'].values
groups = data['client_hash_id']

preprocess = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), X_cols_categorical)], remainder='passthrough')

def run_split(splitter, use_groups):
    preds = np.zeros(len(data))
    splits = splitter.split(X, y, groups=groups) if use_groups else splitter.split(X, y)
    for train_idx, test_idx in splits:
        pipe = Pipeline([('prep', preprocess), ('model', RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1))])
        pipe.fit(X.iloc[train_idx], y[train_idx])
        preds[test_idx] = pipe.predict(X.iloc[test_idx])
    return preds

print("Running BEFORE (naive random KFold, ignores client grouping)...")
before_preds = run_split(KFold(n_splits=5, shuffle=True, random_state=42), use_groups=False)

print("Running AFTER (honest GroupKFold by client, from ML-08)...")
after_preds = run_split(GroupKFold(n_splits=5), use_groups=True)

comparison = pd.DataFrame([
    ['BEFORE: random KFold (naive)', mean_absolute_error(y, before_preds), r2_score(y, before_preds), spearmanr(y, before_preds).correlation],
    ['AFTER: GroupKFold by client (honest)', mean_absolute_error(y, after_preds), r2_score(y, after_preds), spearmanr(y, after_preds).correlation],
], columns=['split_strategy', 'MAE', 'R2', 'spearman'])
print("\n" + comparison.to_string(index=False))8

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Running BEFORE (naive random KFold, ignores client grouping)...
Running AFTER (honest GroupKFold by client, from ML-08)...

                      split_strategy      MAE        R2  spearman
        BEFORE: random KFold (naive) 0.008635  0.013197 -0.011936
AFTER: GroupKFold by client (honest) 0.007734 -0.027999  0.058006


In [3]:
# Robustness check: does the naive-split result hold with a different random seed?
before_preds_seed2 = run_split(KFold(n_splits=5, shuffle=True, random_state=7), use_groups=False)
print("BEFORE (naive KFold, seed=7):")
print(f"  MAE: {mean_absolute_error(y, before_preds_seed2):.6f}")
print(f"  R2: {r2_score(y, before_preds_seed2):.6f}")
print(f"  Spearman: {spearmanr(y, before_preds_seed2).correlation:.6f}")
print(f"\nFor comparison, seed=42 gave MAE={mean_absolute_error(y, before_preds):.6f}, R2={r2_score(y, before_preds):.6f}")

# Also check: is client_hash_id ever used as a feature? (it shouldn't be)
print(f"\nIs client_hash_id in the feature columns? {'client_hash_id' in (X_cols_numeric + X_cols_categorical)}")

BEFORE (naive KFold, seed=7):
  MAE: 0.008617
  R2: 0.013017
  Spearman: -0.008165

For comparison, seed=42 gave MAE=0.008635, R2=0.013197

Is client_hash_id in the feature columns? False


## 3. Leakage audit

**Running the full checklist against my ML-08 feature set (avg_position, word_count,
search_volume, competition, age_days, content_type → predicting ctr_observed):**

- [x] **Timeline drawn** — all five features are properties known as of March 2026 or earlier
  (position/CTR measured concurrently; word_count, search_volume, competition are static
  content/keyword attributes; age_days is a past fact). None reach into a future window.
- [x] **No label-derived or sibling columns** — none of the five features are computed from
  `gsc_clicks` or `gsc_impressions`, confirmed by construction and re-checked below.
- [x] **No product-decision flags used** — confirmed in ML-04: the real warehouse release
  contains no composite health-score or triage-flag columns at all (unlike the paper's internal
  data, which does have Health Score / Optimization Flags). Nothing to accidentally leak here.
- [ ] **Population selection checked for outcome-window information — FOUND A REAL ISSUE.**
  My filter uses `d.is_published IS TRUE AND d.is_deleted IS NOT TRUE`. These are **current
  snapshot-time flags**, not "as of March 2026" flags — the same category of problem as the
  `content_updated_date` future-dating issue found back in ML-04. A page later deleted or
  unpublished (after March, before the snapshot was taken) is silently excluded from my
  analysis, even though it was live and generating real clicks in March. This is a form of
  survivorship bias — it directly parallels the paper's own caution in Finding #8 about the
  small `365+ x 361+` cell being distorted by "the active-content subset introduces strong
  survivor bias." Quantified below.
- [x] **Split grouped by client** — GroupKFold by `client_hash_id`, verified zero overlap
  across all 5 folds in ML-08 and reused in Section 2 above.
- [x] **Base rate printed next to every metric** — the position-tier baseline sits alongside
  every model metric throughout ML-07 and ML-08, never a bare model score alone.
- [x] **Top feature importance sanity-checked** — `word_count` (51%) was investigated, not
  just celebrated; ML-08's error analysis confirmed the largest errors trace to 1-impression
  rows, not to a suspiciously perfect leaky feature.
- [x] **Metrics recomputed out-of-fold** — every MAE/R2/Spearman number in ML-07, ML-08, and
  Section 2 above comes from out-of-fold predictions, never in-sample.
  **Quantified:** only 0.1% of March-active rows (3,770 of 3,611,061) are affected by this
current-snapshot-status filter — 1,817 later-deleted, 1,953 later-unpublished. The
survivorship-bias concern is real in principle and correctly flagged, but at this scale it
is not practically distorting the feature frame or the model results above. Worth naming in
the data contract regardless, since a future release with heavier churn could make this matter
more than it does here.

In [4]:

# Quantify the survivorship-bias concern: how many March-2026-active pages does the
# current-snapshot is_published/is_deleted filter actually remove?
survivorship_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows_before_publish_filter,
        SUM(CASE WHEN d.is_published IS TRUE AND d.is_deleted IS NOT TRUE THEN 1 ELSE 0 END) AS kept_after_filter,
        SUM(CASE WHEN d.is_deleted IS TRUE THEN 1 ELSE 0 END) AS excluded_as_deleted,
        SUM(CASE WHEN d.is_published IS NOT TRUE THEN 1 ELSE 0 END) AS excluded_as_unpublished
    FROM {FACT} f
    JOIN {DIM_CONTENT} d ON f.content_hash_id = d.content_hash_id AND f.client_hash_id = d.client_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
""").df()
print(survivorship_check.to_string(index=False))
pct_excluded = 100 * (1 - survivorship_check['kept_after_filter'][0] / survivorship_check['total_march_rows_before_publish_filter'][0])
print(f"\n{pct_excluded:.1f}% of March-active rows excluded by a CURRENT (not March) publish/delete status.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_march_rows_before_publish_filter  kept_after_filter  excluded_as_deleted  excluded_as_unpublished
                                3611061          3609108.0               1817.0                   1953.0

0.1% of March-active rows excluded by a CURRENT (not March) publish/delete status.


## 4. Claim rewrite

**Original claim (from my own ML-08 writeup):** "Random Forest wins on MAE but loses on the
ranking metric this lane actually depends on."

**Why this needs a rewrite:** "wins" and "loses" are sports-score language implying a decisive,
generalizable verdict. What I actually have is one out-of-fold comparison, on one month of
data, from one client population (46 of 104 total clients) — informative, but not strong enough
to support words that sound like a settled contest.

**Rewritten, safe version:** "On this March-2026 slice, Random Forest showed a lower out-of-fold
MAE than the position-tier baseline (0.0077 vs 0.0085), but a weaker Spearman rank correlation
(0.057 vs 0.166). Since this lane's actual output is a ranked list, the baseline's stronger
ranking performance is the more decision-relevant number here — directionally, the added
features do not yet show a clear ranking advantage over the simpler baseline, on this slice."

**Second claim needing a rewrite (from Section 2 above):** "R² being higher under the naive
split is consistent with what leakage usually looks like."

**Rewritten:** "R² was higher under the naive split, which is the *direction* a leakage effect
would produce — though MAE and Spearman moved the opposite way under the same comparison, so
this dataset does not offer a clean, one-directional confirmation of a leakage effect. Grouped
splitting remains the correct, honest default regardless of this ambiguity, since it is the
only version of the two that guarantees no client appears in both train and test."

In [5]:
# Recompute the exact numbers the rewritten claims above reference, so the claim
# is backed by code in THIS notebook, not just asserted in prose.

# Position-tier baseline, out-of-fold, same honest GroupKFold-by-client split used for after_preds
def position_tier(p):
    if p <= 3: return '1_top_3'
    elif p <= 10: return '2_page_1'
    elif p <= 20: return '3_striking'
    elif p <= 50: return '4_page_3_5'
    else: return '5_deep'

data['position_tier'] = data['avg_position'].apply(position_tier)
baseline_oof = np.zeros(len(data))

for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups=groups):
    tier_lookup = data.iloc[train_idx].groupby('position_tier')['ctr_observed'].mean()
    baseline_oof[test_idx] = data.iloc[test_idx]['position_tier'].map(tier_lookup).fillna(tier_lookup.mean())

claim_check = pd.DataFrame([
    ['baseline_position_tier', mean_absolute_error(y, baseline_oof), spearmanr(y, baseline_oof).correlation],
    ['random_forest (honest split)', mean_absolute_error(y, after_preds), spearmanr(y, after_preds).correlation],
], columns=['method', 'MAE', 'spearman'])

print("Numbers backing the rewritten claim above:")
print(claim_check.to_string(index=False))


Numbers backing the rewritten claim above:
                      method      MAE  spearman
      baseline_position_tier 0.008499  0.166072
random_forest (honest split) 0.007734  0.058006


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.